# Study 900 — Quality-Income 💎

**Does screening dividends for *quality* beat *chasing yield*?**

High dividend **yield** is a notorious value-trap magnet — the fattest yields often mark
distressed payers about to cut. **Quality**-dividend screens (durable, growing payers)
were sold as the fix. We race a **quality sleeve** (SCHD + NOBL) against a **raw
high-yield sleeve** (SPHD + VYM) and against **SPY**, all on monthly total returns, all
measured **excess of cash** (minus BIL), over the common window
2013-11 → 2026-06 (152 months, NOBL-bound).

*Numbers below are the frozen headline (`docs/results.md`); the live cells run the fast
synthetic control. Short-history / survivor caveat: young ETFs, one mostly-bull regime.*


## 1. The idea in one line

A stock can yield 9% because it's cheap and durable — or because the market knows the dividend is about to be cut. **Yield** screens can't tell the difference; **quality** screens (25 straight years of raises, strong cash flow) try to. If the value-trap story is real, the quality sleeve should ride *smoother* — especially through the years when traps blow up.

In [1]:
R = {'start': '2013-11', 'end': '2026-06', 'n_months': 152, 'q_cagr': 11.0, 'q_vol': 13.9, 'q_sharpe': 0.697, 'q_maxdd': -22.4, 'q_wealth': 3.75, 'y_cagr': 10.13, 'y_vol': 13.7, 'y_sharpe': 0.65, 'y_maxdd': -27.5, 'y_wealth': 3.4, 'spy_cagr': 14.02, 'spy_vol': 14.5, 'spy_sharpe': 0.862, 'spy_maxdd': -23.9, 'spy_wealth': 5.27, 'gap': 0.047, 'diff_bps': 6.7, 'diff_ann': 0.81, 't_1s': 0.77, 't_nw': 0.57, 'ci_lo': -0.191, 'ci_hi': 0.241, 'p_neg': 0.355, 'era_e_gap': -0.007, 'era_e_diff': 0.6, 'era_e_t': 0.46, 'era_e_n': 74, 'era_l_gap': 0.06, 'era_l_diff': 1.0, 'era_l_t': 0.41, 'era_l_n': 78, 'qspy_gap': -0.165, 'qspy_diff': -2.81, 'qspy_t': -1.45, 'yspy_gap': -0.212, 'yspy_diff': -3.62, 'yspy_t': -1.44, 'cost3_q': 0.697, 'cost3_y': 0.65, 'cost10_q': 0.696, 'cost10_y': 0.65, 'drag3': 0.2, 'drag10': 0.7, 'turn': 0.6, 'null_mean_t': -0.4, 'null_sd_t': 1.03, 'null_fire': 1, 'planted_gap': 0.505, 'planted_t': 3.03, 'planted_diff': 6.59, 'cal_years': [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], 'cal_q': [-4.4, 27.4, 11.7, 27.7, -4.9, 6.3, 9.2, 5.6], 'cal_y': [-6.0, 22.2, -4.7, 25.5, 0.1, 3.9, 17.9, 9.3], 'cal_spy': [-4.6, 31.2, 18.3, 28.7, -18.2, 26.2, 24.9, 17.7]}
print('QUALITY (SCHD+NOBL): CAGR %5.2f%%  exSharpe %.3f  maxDD %.1f%%'
      % (R['q_cagr'], R['q_sharpe'], R['q_maxdd']))
print('YIELD   (SPHD+VYM) : CAGR %5.2f%%  exSharpe %.3f  maxDD %.1f%%'
      % (R['y_cagr'], R['y_sharpe'], R['y_maxdd']))
print('SPY                : CAGR %5.2f%%  exSharpe %.3f  maxDD %.1f%%'
      % (R['spy_cagr'], R['spy_sharpe'], R['spy_maxdd']))

QUALITY (SCHD+NOBL): CAGR 11.00%  exSharpe 0.697  maxDD -22.4%
YIELD   (SPHD+VYM) : CAGR 10.13%  exSharpe 0.650  maxDD -27.5%
SPY                : CAGR 14.02%  exSharpe 0.862  maxDD -23.9%


## 2. Where quality actually wins — the crisis years

The whole thesis lives in the **stress years**, where yield-traps get punished. Look at 2020 (COVID) and 2022 (rate shock):

In [2]:
for yr, q, y, s in zip(R['cal_years'], R['cal_q'], R['cal_y'], R['cal_spy']):
    star = '  <-- yield sleeve cratered' if (y < 0 and q > 5) else ''
    print(f'{yr}:  quality {q:+6.1f}%   yield {y:+6.1f}%   SPY {s:+6.1f}%{star}')

2018:  quality   -4.4%   yield   -6.0%   SPY   -4.6%
2019:  quality  +27.4%   yield  +22.2%   SPY  +31.2%
2020:  quality  +11.7%   yield   -4.7%   SPY  +18.3%  <-- yield sleeve cratered
2021:  quality  +27.7%   yield  +25.5%   SPY  +28.7%
2022:  quality   -4.9%   yield   +0.1%   SPY  -18.2%
2023:  quality   +6.3%   yield   +3.9%   SPY  +26.2%
2024:  quality   +9.2%   yield  +17.9%   SPY  +24.9%
2025:  quality   +5.6%   yield   +9.3%   SPY  +17.7%


In **2020** the quality sleeve returned **+11.7%** while the yield sleeve fell **−4.7%** (SPHD's high-yield names got crushed). That single dodge is why quality's worst drawdown (**−22.4%**) is ~5 points shallower than yield's (**−27.5%**). *But* yield's low-vol screen actually *won* 2022 and 2024 — it's a two-way trade.

## 3. Is it just luck? A live synthetic control

We plant a real quality-over-yield edge in a seeded toy world (`edge>0`) and check the detector recovers it — and stays *silent* on the null (`edge=0`, no advantage). No network.

In [3]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from quality_income import data, strategy as st
null = st.synthetic_detect(data.synthetic_world(n_months=150, edge=0.0, seed=900))
planted = st.synthetic_detect(data.synthetic_world(n_months=150, edge=0.03, seed=900))
print('null world   : gap NW t = %+.2f  (should be ~0)' % null['t_nw'])
print('planted world: gap NW t = %+.2f  (should light up)' % planted['t_nw'])

null world   : gap NW t = +1.65  (should be ~0)
planted world: gap NW t = +3.03  (should light up)


## 4. The honest verdict

On the real tape the quality sleeve edges the yield sleeve by only **+0.9 pp/yr** of return and a Sharpe gap of **+0.047** — and that gap is **not** statistically distinguishable from zero (HAC *t* = **+0.57**, bootstrap 95% CI **[-0.19, +0.24]** straddles zero, P(quality behind) = 0.35). What quality *does* deliver is a real **drawdown cushion** — dodging the yield-trap blowups — not a certified return premium. And **both** dividend sleeves trailed plain **SPY** by ~3-4 pp/yr. **Signal: Weak** (a real trap-avoidance profile, an insignificant Sharpe edge), **Tradability: Fragile** (cheaply buyable, but a risk profile, not an edge).